# 00. Data Pipeline Overview

This notebook maps the `src/data` pipeline from config to files to datamodule setup. It is meant to answer: **what data config is being used, where is the `.h5ad`, and what objects get built before a dataloader exists?**

For the Replogle sampling path, the high-level flow is:

```text
Hydra cfg
  -> data config: configs/data/replogle_finetune.yaml
  -> DataModule class: Tahoe100mPerturbationDataModule for ReplogleFinetune
  -> setup(): scan metadata and build global dictionaries
  -> setup_dataset(): split indices and construct H5adSentenceDataset objects
  -> val_dataloader()/train_dataloader(): wrap dataset with CellSetBatchSampler
```

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=None):
    cur = Path(start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / "src" / "data").exists() and (p / "configs").exists():
            return p
    raise RuntimeError("Could not find PerturbDiff repo root")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_NAME = "replogle"
PERTURB_ROOT = PROJECT_ROOT / "data" / DATASET_NAME / "perturb_data"
RUN_DIR = PROJECT_ROOT / "runs" / "data_pipeline_overview"
WANDB_DIR = PROJECT_ROOT / "wandb"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PERTURB_ROOT:", PERTURB_ROOT)
assert PERTURB_ROOT.exists(), "Run example/replogle/00_download_data.ipynb first."

## Compose the Same Data Config

Hydra merges `configs/rawdata_diffusion_sampling.yaml`, `configs/data/replogle_finetune.yaml`, `configs/path/trixie_path.yaml`, and the overrides below into one `cfg` object.

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

overrides = [
    "path=trixie_path",
    f"path.tmp_dir={PERTURB_ROOT}",
    f"path.diffusion.save_dir={RUN_DIR}",
    f"path.wandb.logging_dir={WANDB_DIR}",
    "data=replogle_finetune",
    "data.normalize_counts=10",
    "data.pad_length=2000",
    "data.embed_key=X_hvg",
    "model.hidden_num=[2000,512]",
    "model.input_dim=2000",
    "optimization.micro_batch_size=16",
    "data.use_cell_set=8",
]

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "configs"), version_base=None):
    cfg = compose(config_name="rawdata_diffusion_sampling", overrides=overrides)
OmegaConf.resolve(cfg)

print("data_name:", cfg.data.data_name)
print("dataset_name:", cfg.data.dataset_name)
print("dataset_path:", cfg.data.dataset_path)
print("selected_gene_file:", cfg.data.selected_gene_file)
print("indices_cache_dir:", cfg.data.indices_cache_dir)
print("pert_col:", cfg.data.pert_col)
print("cell_line_key:", cfg.data.cell_line_key)
print("batch_col:", cfg.data.perturbseq_batch_col)
print("control_pert:", cfg.data.control_pert)
print("use_cell_set:", cfg.data.use_cell_set)
print("micro_batch_size:", cfg.optimization.micro_batch_size)

## Inspect the Physical `.h5ad` Layout

The project reads `.h5ad` files directly with `h5py` for speed. `metadata_cache.py` reads categorical obs arrays; `dataset_io.py` reads expression rows from `obsm/{embed_key}` or `X`.

In [ ]:
import h5py
from src.common.utils import safe_decode_array

with h5py.File(cfg.data.dataset_path, "r") as f:
    print("top-level keys:", list(f.keys()))
    print("obs keys sample:", list(f["obs"].keys())[:20])
    print("obsm keys:", list(f.get("obsm", {}).keys()))
    print("var keys sample:", list(f.get("var", {}).keys())[:20])
    for col in [cfg.data.pert_col, cfg.data.cell_line_key, cfg.data.perturbseq_batch_col]:
        obj = f["obs"][col]
        print("\nobs column:", col)
        if hasattr(obj, "keys") and "categories" in obj:
            cats = safe_decode_array(obj["categories"][:])
            codes = obj["codes"][:]
            print("  categorical categories:", len(cats), cats[:8])
            print("  codes shape:", codes.shape, "min/max:", codes.min(), codes.max())
        else:
            raw = obj[:]
            print("  raw shape:", raw.shape, "dtype:", raw.dtype, "example:", raw[:8])
    emb = f["obsm"][cfg.data.embed_key]
    print("\nobsm embedding key:", cfg.data.embed_key)
    print("  type:", type(emb))
    print("  attrs:", dict(emb.attrs))
    if hasattr(emb, "shape"):
        print("  shape:", emb.shape)
    elif "shape" in emb.attrs:
        print("  sparse shape attr:", emb.attrs["shape"])

## Selected Genes

`selected_gene_file` defines the 2,000-gene working space. If `embed_key` is `X_hvg`, the code usually reads `obsm/X_hvg` directly. Otherwise it can read `X` and then mask/reorder to these genes.

In [ ]:
import pickle

with open(cfg.data.selected_gene_file, "rb") as fin:
    selected_genes = pickle.load(fin)
if isinstance(selected_genes, set):
    selected_genes = sorted(selected_genes)
print("selected gene count:", len(selected_genes))
print("first genes:", selected_genes[:10])

## Datamodule Class Selection

`src/apps/sampling/sampling_setup.py` routes `ReplogleFinetune` through `Tahoe100mPerturbationDataModule`. The name is historical: its split logic works for Tahoe-style and Replogle-style perturb-seq fine-tuning data.

In [ ]:
from src.apps.sampling.sampling_setup import build_sampling_datamodule
from src.common.utils import setup_loggings

logger = setup_loggings(cfg)
dm = build_sampling_datamodule(cfg, logger)

print("datamodule class:", type(dm).__name__)
print("all_split_names:", dm.all_split_names)
print("dataset_name_list:", dm.dataset_name_list)
print("file_list:", dm.file_list)
print("data_type_list:", dm.data_type_list)
print("# perturbations:", len(dm.pert_dict))
print("# cell types:", len(dm.cell_type_dict))
print("# batches:", len(dm.batch_dict))
print("example perturbations:", list(dm.pert_dict.items())[:8])
print("cell type dict:", dm.cell_type_dict)
print("example batch dict:", list(dm.batch_dict.items())[:8])

At this point only metadata dictionaries exist. The actual `H5adSentenceDataset` objects are created later by `dm.setup_dataset()`, covered in the next notebooks.